# 4.6c Graded visualisation 3: a distribution before and after the positive test

**Notebook 4.6 is split in four:** [a: move and Benji](roos_04.6a-move-and-baby.ipynb) (§4.6.1–4.6.9) · [b: symptoms and topics](roos_04.6b-symptoms-and-topics.ipynb) (§4.6.10–4.6.11) · [c: graded visualisation 3](roos_04.6c-graded-vis3-positive-test.ipynb) (§4.6.12) · [d: the hard weeks](roos_04.6d-hard-weeks.ipynb) (§4.6.13). Loading, the four dates, `save` and the helpers §4.6.13 reuses live in [roos_046_shared.py](roos_046_shared.py).

In [ ]:
import re
import unicodedata
from collections import Counter

import numpy as np
import pandas as pd
from goad_toolkit.analytics import NullDistribution
from goad_toolkit.visualizer import (
    PlotSettings,
)
from roos_046_shared import (
    EVENT_DATE,
    SIDE_COLORS,
    load_own,
    local_chat,
    save,
)
from scipy import optimize, stats

from scripts.plots import BarPlot
from scripts.roos_plots import BeforeAfterDensityPlot


## Loading your data

Same chat as 4.6a: `load_own()` (in `roos_046_shared.py`) loads the export, merges Louis's three names into one, and adds `n_chars`.

In [ ]:
own = load_own()
own.head()

## 4.6.12 Graded visualisation 3: a distribution before and after the positive test

**The assignment (goal 4.17):** *"de eigen chat splitsen op een bekend event (uit 03.3),
berichten per dag en gaps binnen bursts aan beide kanten fitten, en het resultaat opschrijven
als zin met parameters erin."* And the theme's bar: *"welke familie past, en waarom hoort dat
mechanisme bij jouw chat?"* So every fit below comes with a family **and** a mechanism.

**My hypothesis**, from earlier exploration: *the pregnancy changed what we talked about, not
how we talked.* Two layers test it:

1. **How we talked** (the rhythm): gaps between messages inside a burst, and messages per day.
   Expectation: barely any change.
2. **What we talked about**: messages that mention fatigue or nausea. Expectation: a clear rise.

**Focus.** Messages per day is required by the assignment and is fitted below, but it is a
side layer: how many messages we send is not what this story is about. The weight sits on the
gaps within bursts and on the content, and §4.6.13 zooms in on the part that matters most to
me: the hard weeks of the first trimester.

**Event and windows.** The positive pregnancy test, **29 August 2024**, with **77 days (11
weeks) on each side**: before = 13 June – 28 August, after = 29 August – 13 November 2024. Both
windows sit inside the long-distance period (Louis in Paris, me in the Netherlands; he moved to
Utrecht mid-November 2024), which keeps the move out of the comparison. Whole weeks on both
sides also means both windows hold the same number of Mondays, Saturdays, and so on.

**Choices taken from my own notebooks, not invented here:**

- **Event date `2024-08-29`**, verified in the chat itself, and the same as `my_event` in
  `roos_04.4`. `roos_03.3`'s marker for the test sits on `2024-09-01`, three days late: that
  date is actually the start of pregnancy week 5 (see §4.6.13), so the 03.3 marker is the one
  to correct.
- **A burst ends at a gap of more than 60 minutes**, any sender: the same 1-hour cut `roos_04.4`
  uses for its gap fit ("a burst, not overnight"). Note this is a *different* "burst" from
  §4.6.6, where a burst is a run of messages from one author. 15 and 30 minutes are tried in
  the robustness check.
- **Colours**: grey for *before*, crimson for *after*, in every chart below: the grey-first rule
  from 02.2, where the side that carries the change gets the colour.

In [ ]:
# EVENT_DATE (positive test, 2024-08-29), SIDE_COLORS, save() and chat (local clock) come from
# roos_046_shared.py, so 4.6d uses exactly the same ones.
WINDOW_DAYS = 77  # 11 weeks each side: whole weeks, so both sides hold the same weekdays
BURST_GAP_MINUTES = 60  # same 1-hour cut roos_04.4 uses for "a burst, not overnight"
NIGHT_START_HOURS = range(0, 7)  # cleaning only: drop gaps that START between 00:00 and 06:59
SIDES = ["before", "after"]

chat = local_chat(own)


def window(days: int = WINDOW_DAYS) -> tuple[pd.Timestamp, pd.Timestamp]:
    """[start, end): `days` days before the event and `days` days from it onwards."""
    return EVENT_DATE - pd.Timedelta(days=days), EVENT_DATE + pd.Timedelta(days=days)


start, end = window()
print(f"before: {start.date()} to {(EVENT_DATE - pd.Timedelta(days=1)).date()}")
print(f"after:  {EVENT_DATE.date()} to {(end - pd.Timedelta(days=1)).date()}")
print(f"{chat['ts_local'].between(start, end, inclusive='left').sum():,} messages in the two windows")

### Step 1 — messages per day (required; a side layer)

**What:** count the messages on every calendar day in each window, *including days with zero
messages* (a silent day is an observation, not a missing one). Then fit two families by maximum
likelihood and compare them with AIC (`2k − 2·log-likelihood`; lower is better, and a
difference of more than about 10 is decisive).

- **Poisson**: messages arrive at one steady rate μ. It makes a checkable promise: variance =
  mean, so variance/mean ≈ 1.
- **Negative binomial**: a Poisson whose rate itself changes from day to day. Two parameters:
  the mean μ and a shape r (or dispersion α = 1/r). Small r = the daily rate wanders a lot.

**Why not `DistributionFitter` here:** it passes `nbinom` to `scipy.stats.fit`, which only
tries whole-number values for r. A dispersion of 1.4 would come back as 1, so the fit is done
directly with `scipy.optimize` instead (same maximum-likelihood idea, r allowed to be any
positive number).

In [ ]:
def messages_per_day(days: int = WINDOW_DAYS) -> pd.DataFrame:
    """One row per calendar day in the window, zero-message days included."""
    lo, hi = window(days)
    in_window = chat[(chat["ts_local"] >= lo) & (chat["ts_local"] < hi)]
    all_days = pd.date_range(lo, hi - pd.Timedelta(days=1), freq="D")
    counts = (
        in_window.groupby(in_window["ts_local"].dt.normalize()).size()
        .reindex(all_days, fill_value=0)
        .rename("messages").rename_axis("day").reset_index()
    )
    counts["side"] = np.where(counts["day"] < EVENT_DATE, "before", "after")
    return counts


def aic(log_likelihood: float, k: int) -> float:
    return 2 * k - 2 * log_likelihood


def nbinom_from(mu: float, r: float):
    """scipy's nbinom takes (n, p); n is r, and p = r / (r + mu)."""
    return stats.nbinom(r, r / (r + mu))


def fit_daily(values: np.ndarray) -> dict:
    """Poisson and negative binomial by maximum likelihood, compared on AIC.

    Not via DistributionFitter: it hands nbinom to scipy.stats.fit, which only tries
    whole-number r -- a dispersion of 2.3 would come back as 2.
    """
    poisson_mu = values.mean()  # the Poisson MLE is the sample mean
    poisson_ll = stats.poisson.logpmf(values, poisson_mu).sum()

    def nll(theta):  # log-parameters keep mu and r positive
        return -nbinom_from(np.exp(theta[0]), np.exp(theta[1])).logpmf(values).sum()

    res = optimize.minimize(nll, [np.log(values.mean()), 0.0], method="Nelder-Mead",
                            options={"xatol": 1e-8, "fatol": 1e-8})
    mu, r = np.exp(res.x)
    return {
        "n_days": len(values),
        "mean": values.mean(),
        "vmr": values.var(ddof=1) / values.mean(),
        "poisson_mu": poisson_mu,
        "poisson_aic": aic(poisson_ll, 1),
        "nb_mu": mu,
        "nb_r": r,
        "nb_alpha": 1 / r,
        "nb_aic": aic(-res.fun, 2),
        "nb_dist": nbinom_from(mu, r),
    }


per_day = messages_per_day()
daily = {side: fit_daily(per_day.loc[per_day["side"] == side, "messages"].to_numpy().astype(float))
         for side in SIDES}

for side in SIDES:
    d = daily[side]
    print(f"{side:6s} {d['n_days']} days   observed mean {d['mean']:5.1f}   variance/mean {d['vmr']:5.1f}")
    print(f"       poisson  mu = {d['poisson_mu']:5.1f}                      AIC {d['poisson_aic']:8.1f}")
    print(f"       nbinom   mu = {d['nb_mu']:5.1f}   r = {d['nb_r']:.2f}   alpha = {d['nb_alpha']:.2f}   AIC {d['nb_aic']:8.1f}"
          f"   (dAIC poisson - nbinom = {d['poisson_aic'] - d['nb_aic']:,.0f})")

**What this means.**

- **Poisson is rejected on both sides, by a wide margin**: ΔAIC ≈ 2,400. Variance/mean is
  about 32 on both sides, where Poisson promises 1. That rejection *is* a finding: there is no
  steady daily rate in this chat.
- **The negative binomial is the family**, on both sides.
- **The mean barely moved:** μ ≈ 85 before, ≈ 96 after.
- **The dispersion moved more than the mean:** r = 1.38 before and 2.02 after, so the
  before-days were *more* erratic. The robustness check below shows where that comes from.

**Mechanism: why a negative binomial belongs to this chat.** A negative binomial is a
*gamma-Poisson mixture*. Each day has its own rate, drawn from a spread of possible rates
(the gamma part). Given that rate, messages arrive at random moments (the Poisson part).
For us, what sets a day's rate is easy to name:

- a workday or a weekend;
- whether there are plans to make;
- mood;
- above all, whether we are *apart* (long-distance, so the chat is the relationship) or
  *together* on a visit (and the chat goes nearly quiet).

A day is not "85 messages give or take a few". It is "a rate of about 85 that routinely
halves or doubles", which is exactly what r ≈ 1.4–2 says.

One more thing worth knowing: before the test, the variance/mean check and the fitted r don't
agree as well as after. That hints the before window holds two kinds of day (ordinary days and
near-empty ones). See the weekly table in the robustness section.

In [ ]:
daily_settings = PlotSettings(
    figsize=(10, 5),
    title="",
    xlabel="messages per day",
    ylabel="share of days, per message/day",
)
top = int(np.ceil(per_day["messages"].max() / 20) * 20)
fig, ax = BeforeAfterDensityPlot(daily_settings).plot(
    samples={side: per_day.loc[per_day["side"] == side, "messages"].to_numpy() for side in SIDES},
    bins=np.arange(0, top + 20, 20),
    colors=SIDE_COLORS,
    densities={side: daily[side]["nb_dist"].pmf for side in SIDES},
    discrete=True,
    labels={
        side: f"{side} the test: μ = {daily[side]['nb_mu']:.0f}, r = {daily[side]['nb_r']:.1f}\n"
              f"(negative binomial, {daily[side]['n_days']} days)"
        for side in SIDES
    },
    label_positions={"before": (0.55, 0.92), "after": (0.55, 0.76)},
)
ax.set_title(
    f"Messages per day: a small rise (μ {daily['before']['nb_mu']:.0f} → {daily['after']['nb_mu']:.0f}), "
    f"inside the day-to-day spread",
    loc="left", fontsize=12,
)
fig.text(0.01, -0.04, "Bars: observed days (bins of 20 messages). Curves: fitted negative binomial. "
         "Both windows fall in the long-distance period, before the move to Utrecht.",
         fontsize=8.5, color="dimgray")
save(fig, "vis3_messages_per_day")

Bars and curves share one density scale (share of days per message/day), so the eye can
check the fit. The grey curve peaks further left than the crimson one, pulled there by the
handful of near-empty days before the test.

**The hypothesis on this layer:** the rise from μ 85 to 96 is small next to the day-to-day
spread. The shuffle test in step 3 checks whether a random split of the same days would
produce a gap this size anyway.

### Step 2 — gaps within bursts

**What:** the seconds between two consecutive messages, whoever sent them, as long as the gap
is at most the burst threshold (60 minutes). A gap belongs to the side where its first message
was sent.

**Cleaning, not a finding:** gaps that *start* between 00:00 and 06:59 are left out, so a
"good night … good morning" pair doesn't enter as a 7-hour gap. This is only there to keep
sleep out of the measurement. It is not part of the story, and the story below never leans
on it.

**Families compared on AIC:** exponential, lognormal and Weibull, each with location fixed at 0.
Plus a **power-law tail**, p(x) ∝ x^−α above a cut-off x_min. x_min is the value that makes
the fitted tail closest to the data (the smallest Kolmogorov-Smirnov distance, the standard
Clauset-Shalizi-Newman recipe), and α is fitted by maximum likelihood.

**Truncation:** a gap inside a burst *can't* be longer than 60 minutes; the burst definition
cuts it off. A family fitted as if the tail continued beyond that would be judged on data it
was never allowed to see. So every fit here is **truncated at the threshold**: the density is
re-normalised to add up to 1 over the range the data can actually take. (Timestamps are whole
seconds, so the shortest possible gap is 1 s; the very first bins are coarse because of that.)

In [ ]:
def within_burst_gaps(lo: pd.Timestamp, hi: pd.Timestamp, burst_minutes: int = BURST_GAP_MINUTES) -> pd.DataFrame:
    """Seconds between consecutive messages (any sender) inside a burst, labelled by side.

    A gap belongs to the window its first message falls in. Kept: 0 < gap <= burst_minutes.
    Dropped (cleaning, not a finding): gaps starting between 00:00 and 06:59.
    """
    gap = chat["ts_local"].diff().dt.total_seconds()
    started = chat["ts_local"].shift()
    frame = pd.DataFrame({"gap": gap, "started": started}).dropna()
    keep = (
        frame["gap"].gt(0)
        & frame["gap"].le(burst_minutes * 60)
        & ~frame["started"].dt.hour.isin(NIGHT_START_HOURS)
        & frame["started"].ge(lo)
        & frame["started"].lt(hi)
    )
    out = frame[keep].copy()
    out["side"] = np.where(out["started"] < EVENT_DATE, "before", "after")
    return out


# Continuous families, all fitted TRUNCATED to (0, upper]: a burst can't contain a gap longer
# than the burst threshold, so the density is renormalised over the range the data could
# take. Each family: (scipy distribution, start values, unpack log-params -> scipy kwargs).
GAP_FAMILIES = {
    "exponential": (stats.expon, lambda x: [np.log(x.mean())],
                    lambda t: {"scale": np.exp(t[0])}),
    "lognorm": (stats.lognorm, lambda x: [np.log(np.log(x).std()), np.log(x).mean()],
                lambda t: {"s": np.exp(t[0]), "scale": np.exp(t[1])}),
    "weibull": (stats.weibull_min, lambda x: [np.log(0.6), np.log(np.median(x))],
                lambda t: {"c": np.exp(t[0]), "scale": np.exp(t[1])}),
}


def fit_truncated(family: str, x: np.ndarray, lower: float, upper: float) -> dict:
    """Maximum likelihood for a family truncated to [lower, upper]."""
    dist, start_values, unpack = GAP_FAMILIES[family]

    def nll(theta):
        kw = unpack(theta)
        mass = dist.cdf(upper, **kw) - dist.cdf(lower, **kw)
        if not np.isfinite(mass) or mass <= 0:
            return np.inf
        return -(dist.logpdf(x, **kw).sum() - len(x) * np.log(mass))

    res = optimize.minimize(nll, start_values(x), method="Nelder-Mead",
                            options={"xatol": 1e-8, "fatol": 1e-8, "maxiter": 5000})
    kw = unpack(res.x)
    frozen = dist(**kw)
    mass = frozen.cdf(upper) - frozen.cdf(lower)
    k = len(res.x)
    return {
        "family": family, "params": kw, "k": k, "loglik": -res.fun, "aic": 2 * k + 2 * res.fun,
        "pdf": lambda v, f=frozen, m=mass: np.where((v >= lower) & (v <= upper), f.pdf(v) / m, 0.0),
        "converged": res.success,
        # median of the model on the range the data can take, comparable to the observed median
        "model_median": frozen.ppf((frozen.cdf(lower) + frozen.cdf(upper)) / 2),
    }


def fit_powerlaw_tail(x: np.ndarray, upper: float) -> dict:
    """Power law p(x) ~ x^-alpha on [x_min, upper], truncated at the burst threshold.

    x_min is the candidate that makes the fitted tail closest to the data (smallest KS
    distance), the Clauset-Shalizi-Newman recipe; alpha by maximum likelihood given x_min.
    """
    def fit_alpha(tail, x_min):
        logs = np.log(tail).sum()

        def nll(a):
            norm = x_min ** (1 - a) - upper ** (1 - a)
            return -(len(tail) * np.log((a - 1) / norm) - a * logs)

        a = optimize.minimize_scalar(nll, bounds=(1.001, 6), method="bounded").x
        return a, -nll(a)

    best = None
    for x_min in np.unique(np.quantile(x, np.linspace(0.05, 0.9, 40)).round()):
        tail = np.sort(x[x >= x_min])
        a, ll = fit_alpha(tail, x_min)
        model_cdf = (x_min ** (1 - a) - tail ** (1 - a)) / (x_min ** (1 - a) - upper ** (1 - a))
        empirical = np.arange(1, len(tail) + 1) / len(tail)
        ks = np.abs(model_cdf - empirical).max()
        if best is None or ks < best["ks"]:
            best = {"x_min": x_min, "alpha": a, "loglik": ll, "ks": ks, "n_tail": len(tail),
                    "aic": 2 * 1 - 2 * ll}
    return best


def fit_gaps(x: np.ndarray, upper: float) -> dict:
    fits = {name: fit_truncated(name, x, 0.0, upper) for name in GAP_FAMILIES}
    tail = fit_powerlaw_tail(x, upper)
    # Same tail, same range: does a (doubly truncated) lognormal beat the power law there?
    tail["lognorm_aic"] = fit_truncated("lognorm", x[x >= tail["x_min"]], tail["x_min"], upper)["aic"]
    return {"fits": fits, "tail": tail, "n": len(x), "median": np.median(x),
            "best": min(fits, key=lambda name: fits[name]["aic"])}


gaps = within_burst_gaps(start, end)
upper = BURST_GAP_MINUTES * 60
gap_fits = {side: fit_gaps(gaps.loc[gaps["side"] == side, "gap"].to_numpy(), upper) for side in SIDES}

for side in SIDES:
    g = gap_fits[side]
    print(f"{side}: {g['n']:,} gaps, observed median {g['median']:.0f} s, best by AIC: {g['best']} "
          f"(its median within the burst range: {g['fits'][g['best']]['model_median']:.0f} s)")
    for name, f in g["fits"].items():
        params = ", ".join(f"{k} = {v:.3g}" for k, v in f["params"].items())
        print(f"   {name:12s} {params:28s} AIC {f['aic']:11,.0f}   dAIC {f['aic'] - g['fits'][g['best']]['aic']:9,.0f}")
    t = g["tail"]
    print(f"   power-law tail: alpha = {t['alpha']:.2f} above x_min = {t['x_min']:.0f} s "
          f"({t['n_tail']:,} gaps, KS {t['ks']:.3f});  tail AIC power law {t['aic']:,.0f} vs lognorm {t['lognorm_aic']:,.0f}")

**What this means.**

- **Exponential is rejected on both sides, overwhelmingly** (ΔAIC ≈ 10,000). Gaps are *not*
  "replies at random moments at one steady rate".
- **Lognormal wins the three-way comparison on both sides**, with almost identical
  parameters: e^μ ≈ 39 s, and σ = 1.88 before against 1.80 after.
- **The power-law tail has α = 1.10 before and 1.21 after**, above roughly 2 minutes
  (153 s before, 111 s after).
- **On that tail, power law and lognormal are a tie:** ΔAIC is 1 to 2, so the data can't tell
  them apart there.

The lognormal is the *best of the three*, not a good description. Its median within the
burst range is 38 s, while the observed median is 27 s. The plot below shows why.

**Mechanism: Barabási (2005), "The origin of bursts and heavy tails in human dynamics".**

- **An exponential gap distribution means a Poisson process:** each reply comes at a random
  moment, independent of the last one.
- **What Barabási found instead,** for email and letters, is a **heavy tail**. People keep a
  mental list of things to do and always pick the most urgent one. Answering the chat is one
  item on that list.
- **When the chat is at the top of the list,** we answer within seconds: the fast core of
  10–40 s gaps, rapid back-and-forth.
- **When something else takes priority** (work, cooking, a meeting), the chat waits until that
  is done. Those waits have no typical length; they spread out with p(x) ∝ x^−α. His
  priority-queue model predicts **α ≈ 1**, and we find **1.1–1.2**.

**Careful phrasing:** lognormal and power law can't be separated on our tail, and the tail
only spans about 1.5 decades (from ~2 minutes to the 60-minute cut). So the claim is
**"heavy-tailed, as Barabási describes"**, not "an exact power law".

**For the hypothesis:** whichever family you pick, before and after are the *same* shape with
the same parameters.

In [ ]:
gap_settings = PlotSettings(
    figsize=(10, 5.5),
    title="",
    xlabel="seconds between two messages inside a burst (log scale)",
    ylabel="share of gaps per log10(seconds)",
)


def gap_label(side: str) -> str:
    g = gap_fits[side]
    p = g["fits"][g["best"]]["params"]
    t = g["tail"]
    return (f"{side} the test ({g['n']:,} gaps)\n"
            f"solid: {g['best']}, e^μ = {p['scale']:.0f} s, σ = {p['s']:.2f}\n"
            f"dashed: tail ∝ x^-{t['alpha']:.2f} above {t['x_min']:.0f} s")


def tail_density(side: str):
    """The power-law tail as a share of ALL gaps, so it sits on the same scale as the bars."""
    g, t = gap_fits[side], gap_fits[side]["tail"]
    a, x_min, share = t["alpha"], t["x_min"], t["n_tail"] / g["n"]
    norm = x_min ** (1 - a) - upper ** (1 - a)
    return lambda v: np.where(v >= x_min, share * (a - 1) * v ** (-a) / norm, np.nan)


# Timestamps are whole seconds, so bin edges sit halfway between whole numbers: every bin
# holds at least one possible value, and the 1-3 s range has no artificial empty bins.
gap_bins = np.unique(np.round(np.geomspace(1, upper, 32))) - 0.5
gap_bins = np.append(gap_bins, upper + 0.5)

host = BeforeAfterDensityPlot(gap_settings)
fig, ax = host.plot(
    samples={side: gaps.loc[gaps["side"] == side, "gap"].to_numpy() for side in SIDES},
    bins=gap_bins,
    colors=SIDE_COLORS,
    densities={side: gap_fits[side]["fits"][gap_fits[side]["best"]]["pdf"] for side in SIDES},
    log_x=True,
    labels={side: gap_label(side) for side in SIDES},
    label_positions={"before": (0.60, 0.97), "after": (0.60, 0.80)},
)
grid = np.geomspace(1, upper, 400)
for side in SIDES:
    ax.plot(grid, tail_density(side)(grid) * grid * np.log(10), color=SIDE_COLORS[side],
            linestyle="--", linewidth=2)
ax.set_title("Inside a conversation, the tempo stayed the same", loc="left", fontsize=12)
fig.text(0.01, -0.04, f"Gaps of 1 s up to the {BURST_GAP_MINUTES}-minute burst threshold; gaps starting "
         "00:00-06:59 left out. Fits truncated at the threshold.",
         fontsize=8.5, color="dimgray")
save(fig, "vis3_gaps_within_bursts")

The two histograms lie on top of each other. On a log x-axis with logarithmic bins, the bar
height is "share of gaps per unit of log10(seconds)", and the curves are converted to that
same scale (density × x × ln 10), so bars and curves can be compared directly.

Read it in two parts:

- **The hump at 10–40 s:** the conversation itself.
- **The long, almost flat shelf from about 2 minutes to the threshold:** the priority-queue
  waits. A shelf that is flat on a log axis is exactly what p(x) ∝ x^−1 looks like, which is
  why the dashed power-law lines follow it and the lognormal (solid) falls away too early.

### Step 3 — the layer that *should* change: fatigue and nausea

**What:** flag every message in the two windows that mentions nausea or fatigue, in French,
Dutch or English. Then count flagged messages per calendar day (zero days included, as in
step 1).

**How the matching works:**

- **Normalise first:** lowercase, straighten apostrophes, remove accents (`écœurée` →
  `ecoeuree`), so one pattern covers every spelling.
- **Word boundaries everywhere**, so that `crevée` matches but `crevettes` (shrimp) doesn't,
  `claquée` but not `claquement`. `plus d'énergie` ("more energy") is not on the list; only the
  negative forms (`pas d'énergie`, `geen energie`, `no energy`) are.
- **These are not the patterns from §4.6.10.** Those (from `roos_03.3`) have no word boundaries
  in the French part, so `crev[ée]e?` also matches `crevettes`. The lists here are the new,
  stricter ones.

The cell **asserts** that none of the named false positives appear in any match, and prints
the most frequent matched terms so the list can be checked by eye.

In [ ]:
def normalise(text: str) -> str:
    """Lowercase, curly apostrophes straightened, accents removed: 'Écœurée' -> 'ecoeuree'."""
    text = str(text).lower().replace("’", "'").replace("œ", "oe")
    return "".join(c for c in unicodedata.normalize("NFKD", text) if not unicodedata.combining(c))


# Patterns run on normalised text, so no accents in them. \b everywhere: 'crevee' must not
# match 'crevettes', 'claquee' not 'claquement'. 'plus d'energie' (more energy) is not in the
# list; only 'pas d'energie' / 'geen energie' / 'no energy' are.
SYMPTOM_PATTERNS = {
    "nausea": re.compile(
        r"\b(?:"
        r"nausees?|nauseeuses?|nauseeux|(?:envie de )?vomir|ecoeuree?s?|barbouillee?s?|"
        r"misselijk(?:heid)?|overgeven|kotsen|braken|"
        r"nausea|nauseous|vomit(?:ing|ed)?|throw(?:ing)? up|feel(?:ing)? sick|morning sickness"
        r")\b"
    ),
    "fatigue": re.compile(
        r"\b(?:"
        r"fatiguee?s?|crevee?s?|epuisee?s?|naze|claquee?s?|j'?\s?ai sommeil|pas d'energie|"
        r"moe|vermoeid|uitgeput|doodop|slaperig|geen energie|"
        r"tired|exhausted|sleepy|knackered|drained|worn out|no energy"
        r")\b"
    ),
}
EXCLUDED_WORDS = ["lessive", "crevette", "claquement", "plus d'energie"]

in_window = chat[(chat["ts_local"] >= start) & (chat["ts_local"] < end)].copy()
in_window["normalised"] = in_window["message"].map(normalise)
for name, pattern in SYMPTOM_PATTERNS.items():
    in_window[name] = in_window["normalised"].str.contains(pattern)
in_window["either"] = in_window["nausea"] | in_window["fatigue"]

# Guard against the false positives named in the brief: none of these words may be inside a match.
matched_terms = Counter(
    m.group(0) for name, pattern in SYMPTOM_PATTERNS.items()
    for text in in_window.loc[in_window[name], "normalised"] for m in pattern.finditer(text)
)
false_positives = [term for term in matched_terms if any(bad in term for bad in EXCLUDED_WORDS)]
if false_positives:
    raise ValueError(f"excluded words inside a match: {false_positives}")
print(in_window[["nausea", "fatigue", "either"]].sum().rename("messages in the two windows"))
print("\nmost frequent matched terms:", matched_terms.most_common(20))

**Check before trusting it:** every frequent term above is a genuine symptom word, and the
assert passed (no `lessive`, `crevette`, `claquement`, `plus d'énergie`). Two things the
keyword match *can't* separate, worth keeping in mind:

- **Ordinary tiredness vs pregnancy tiredness:** "too tired to correct it" counts the same as
  first-trimester exhaustion. This is why fatigue already had a baseline before the test.
- **Having a symptom vs talking about one:** a pasted "week 6: fatigue, morning nausea…" list
  from a pregnancy app counts too. For *this* hypothesis that is fine, because the claim is
  about what we *talked about*.

⚠️ **The next cell prints short snippets around matches, for checking only. Clear its output
before submitting.**

In [ ]:
# CHECK ONLY -- delete this cell's output before submitting. Short snippets around each match,
# never whole messages.
def snippet(text: str, pattern: re.Pattern, width: int = 30) -> str:
    m = pattern.search(text)
    if m is None:
        return ""
    return "…" + text[max(0, m.start() - width): m.end() + width] + "…"


for name, pattern in SYMPTOM_PATTERNS.items():
    sample = in_window[in_window[name]].sample(8, random_state=1)
    print(f"--- {name}")
    for text in sample["normalised"]:
        print("  ", snippet(text, pattern))

Per-day rates on both sides, plus two shuffle tests (`NullDistribution`, as in 04.4 §4.4.3):
shuffle the before/after label across the 154 days 2,000 times and see how often a random
split produces a difference as large as the real one. One test for the symptom rate, and one
for messages per day from step 1, so both layers get the same test.

In [ ]:
symptom_days = (
    in_window.groupby(in_window["ts_local"].dt.normalize())[["nausea", "fatigue", "either"]].sum()
    .reindex(pd.date_range(start, end - pd.Timedelta(days=1), freq="D"), fill_value=0)
    .rename_axis("day").reset_index()
)
symptom_days["side"] = np.where(symptom_days["day"] < EVENT_DATE, "before", "after")
symptom_rates = symptom_days.groupby("side")[["nausea", "fatigue", "either"]].mean().loc[SIDES]
print(symptom_rates.round(2))

null_symptoms = NullDistribution(
    lambda days: days.groupby("side")["either"].mean().pipe(lambda m: m["after"] - m["before"]),
    n_iter=2000, seed=4,
).run(symptom_days, label="side")
null_daily = NullDistribution(
    lambda days: days.groupby("side")["messages"].mean().pipe(lambda m: m["after"] - m["before"]),
    n_iter=2000, seed=4,
).run(per_day, label="side")
print(f"\nfatigue-or-nausea messages/day, after - before: {null_symptoms.observed:+.2f}, two-sided p = {null_symptoms.p_value():.4f}")
print(f"messages/day, after - before:                   {null_daily.observed:+.1f}, two-sided p = {null_daily.p_value():.4f}")

**What this means.**

- **Fatigue-or-nausea mentions roughly quadrupled:** 0.52 → 2.01 messages a day.
- **Nausea went from zero mentions** in the 77 days before the test to 0.45 a day after.
- **The symptom rise is very unlikely to be chance:** p ≈ 0.001 for the symptom rate, the
  smallest value 2,000 shuffles can resolve, so read it as "p < 0.001".
- **The messages-per-day rise could easily be chance:** p ≈ 0.25 at 11 weeks.

That is the hypothesis's pattern: the content moved, the volume didn't distinguishably move.

One caveat about this test: neighbouring days are not independent (a busy week is busy on
several days), so the shuffle is a little optimistic in both directions.

In [ ]:
bars = (
    symptom_rates.rename(columns={"either": "nausea or fatigue"})
    .reset_index().melt(id_vars="side", var_name="mention", value_name="per_day")
)
symptom_settings = PlotSettings(figsize=(9, 5), title="", xlabel="", ylabel="messages per day that mention it")
fig, ax = BarPlot(symptom_settings).plot(
    data=bars, x="mention", y="per_day", hue="side", hue_order=SIDES, palette=SIDE_COLORS,
    order=["nausea or fatigue", "fatigue", "nausea"], saturation=1,
)
ax.set_xlabel("")
for container in ax.containers:
    ax.bar_label(container, fmt="%.2f", padding=3)
ax.legend_.remove()
for side, x_text in zip(SIDES, [0.62, 0.62]):
    ax.text(x_text, 0.92 if side == "before" else 0.82, f"{side} the test (77 days)",
            transform=ax.transAxes, color=SIDE_COLORS[side], fontweight="bold")
before_rate, after_rate = symptom_rates.loc["before", "either"], symptom_rates.loc["after", "either"]
ax.set_title(f"What changed was the content: fatigue and nausea mentions went from "
             f"{before_rate:.2f} to {after_rate:.2f} a day", loc="left", fontsize=12)
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, -0.04, "A message counts once per bar, however many symptom words it contains. "
         "Keyword match in French, Dutch and English.", fontsize=8.5, color="dimgray")
save(fig, "vis3_symptoms_per_day")

This is the one chart where the crimson bar should jump out, and it does. Same colours as
the two rhythm charts, values written on the bars, and a title that states the finding.
`nausea or fatigue` comes first, since it is the measure the result sentence uses.
Fatigue and nausea are shown separately because they behave differently: fatigue has a
before-baseline, nausea starts from zero.

### Step 4 — the parameter table

Every candidate family, both sides, with its parameters, AIC, and n. ΔAIC is measured
against the best family *for that variable and side*. The power-law row is compared only with
the lognormal fitted on the same tail (shown in its parameters column), because it describes a
different subset of the data (the gaps above x_min).

In [ ]:
rows = []
for side in SIDES:
    d = daily[side]
    rows += [
        {"variable": "messages per day", "side": side, "family": "poisson", "parameters": f"μ = {d['poisson_mu']:.1f}",
         "AIC": d["poisson_aic"], "n": d["n_days"]},
        {"variable": "messages per day", "side": side, "family": "nbinom",
         "parameters": f"μ = {d['nb_mu']:.1f}, r = {d['nb_r']:.2f} (α = {d['nb_alpha']:.2f}); variance/mean = {d['vmr']:.1f}",
         "AIC": d["nb_aic"], "n": d["n_days"]},
    ]
    g = gap_fits[side]
    for name, f in g["fits"].items():
        p = f["params"]
        if name == "exponential":
            text = f"mean = {p['scale']:.0f} s"
        elif name == "lognorm":
            text = f"median = {p['scale']:.1f} s, σ = {p['s']:.2f}"
        else:
            text = f"shape = {p['c']:.2f}, scale = {p['scale']:.0f} s"
        rows.append({"variable": "gaps within bursts", "side": side, "family": name + " (truncated)",
                     "parameters": text, "AIC": f["aic"], "n": g["n"]})
    t = g["tail"]
    rows.append({"variable": "gaps within bursts", "side": side, "family": "power-law tail",
                 "parameters": f"α = {t['alpha']:.2f}, x_min = {t['x_min']:.0f} s (lognorm on same tail: AIC {t['lognorm_aic']:,.0f})",
                 "AIC": t["aic"], "n": t["n_tail"]})
param_table = pd.DataFrame(rows)
param_table["ΔAIC"] = param_table["AIC"] - param_table.groupby(["variable", "side", param_table["family"] == "power-law tail"])["AIC"].transform("min")
pd.set_option("display.max_colwidth", 120)
param_table.round({"AIC": 0, "ΔAIC": 0})

### Step 5 — robustness: does the conclusion survive other reasonable choices?

Two choices could have been made differently, so both are varied:

- **The burst threshold:** 15, 30 or 60 minutes.
- **The window length:** 4, 8 or 11 weeks on each side.

For the daily fit, each window also gets the same shuffle test as step 3.

In [ ]:
robust_rows = []
for minutes in [15, 30, 60]:
    g = within_burst_gaps(start, end, minutes)
    fits = {side: fit_gaps(g.loc[g["side"] == side, "gap"].to_numpy(), minutes * 60) for side in SIDES}
    ln = {side: fits[side]["fits"]["lognorm"]["params"] for side in SIDES}
    robust_rows.append({
        "check": f"burst threshold {minutes} min",
        "best family (before / after)": f"{fits['before']['best']} / {fits['after']['best']}",
        "before": f"median {ln['before']['scale']:.0f} s, σ {ln['before']['s']:.2f} (n {fits['before']['n']:,})",
        "after": f"median {ln['after']['scale']:.0f} s, σ {ln['after']['s']:.2f} (n {fits['after']['n']:,})",
        "change": f"median {ln['after']['scale'] / ln['before']['scale'] - 1:+.0%}",
        "shuffle p": "",
    })
for weeks in [4, 8, 11]:
    days = messages_per_day(weeks * 7)
    fits = {side: fit_daily(days.loc[days["side"] == side, "messages"].to_numpy().astype(float)) for side in SIDES}
    best = {side: "nbinom" if fits[side]["nb_aic"] < fits[side]["poisson_aic"] else "poisson" for side in SIDES}
    null = NullDistribution(
        lambda d: d.groupby("side")["messages"].mean().pipe(lambda m: m["after"] - m["before"]),
        n_iter=2000, seed=4,
    ).run(days, label="side")
    robust_rows.append({
        "check": f"window {weeks} weeks each side",
        "best family (before / after)": f"{best['before']} / {best['after']}",
        "before": f"μ {fits['before']['nb_mu']:.0f}, r {fits['before']['nb_r']:.2f} ({weeks * 7} days)",
        "after": f"μ {fits['after']['nb_mu']:.0f}, r {fits['after']['nb_r']:.2f} ({weeks * 7} days)",
        "change": f"μ {fits['after']['nb_mu'] / fits['before']['nb_mu'] - 1:+.0%}",
        "shuffle p": f"{null.p_value():.3f}",
    })
robustness = pd.DataFrame(robust_rows)
robustness

In [ ]:
# Where does the window-dependence come from? Messages per week (Thursday-Wednesday, so the
# last "before" week ends the day before the test), with the number of days under 20 messages.
weekly_volume = (
    per_day.set_index("day")["messages"].resample("W-WED")
    .agg(messages="sum", quiet_days=lambda s: int((s < 20).sum()))
)
weekly_volume.index = weekly_volume.index.date
weekly_volume.rename_axis("week ending").T

**What this means.**

**Gaps within bursts: robust.**

- Lognormal wins on both sides at every threshold.
- The before/after change in the median parameter stays within ±5%.
- The parameters themselves depend on the threshold: a longer cut lets more of the heavy
  tail in, so e^μ goes 28 → 33 → 39 s and σ rises. That is expected for a heavy tail and is
  the same on both sides. (At 30 minutes the numbers match the earlier independent parse:
  e^μ ≈ 33 s, σ 1.71 / 1.64.)
- *The tempo of our conversations did not change* holds however a burst is defined.

**Messages per day: not robust.**

- The family holds (negative binomial everywhere), but the size of the change depends on the
  window: μ +58% at 4 weeks and +37% at 8 weeks (both p ≈ 0.01), against +12% at 11 weeks
  (p ≈ 0.25).
- The weekly table below shows where that comes from: the two weeks ending 14 and 21 August,
  right before the test, are by far the quietest in the window (198 and 246 messages), with
  several near-empty days.
- The shorter the window, the more weight those two weeks get.

**Explained:** we were together in mid-August 2024. Those quiet weeks are exactly the
"together" days from step 1's mechanism (when we are in the same place, the chat goes nearly
silent), not an effect of the test. The 11-week window is the fairer comparison because it
holds that one visit as one of many ups and downs, instead of letting it dominate a short
window. So: *over 11 weeks, daily volume did not change beyond its normal spread; the rise in
shorter windows is the end of a visit, not the pregnancy.*

### What the data can and cannot show

**Remaining confounders, named explicitly:**

- **The end of the summer holidays.** 29 August is roughly when "back to work" starts. A
  return to routine could on its own raise the daily volume (people message more from a desk)
  and the fatigue mentions (work is tiring). The event and the end of summer can't be
  separated with one event in one chat.
- **Time together in mid-August 2024** (confirmed). It explains the window-dependence of the
  daily volume layer, and leaves the gaps and the symptoms untouched.
- **Keyword fatigue is generic.** Part of the fatigue rise could be ordinary tiredness from the
  return to work rather than pregnancy. **Nausea is the cleaner signal:** zero mentions in 11
  weeks before, regular mentions after, and nausea is not something a work week produces.

**Correlation is not causation.**

- **What the data show:** the symptom words rise sharply at the event. The timing fits early
  pregnancy: nausea appears in the first weeks after a positive test.
- **What the data cannot show:** that the pregnancy *caused* the rise (one event, no control
  period with the same calendar), or whether a mention means *having* a symptom or *talking
  about* one.
- **Scope:** n = 2 people, one relationship. Every result describes this chat, not
  pregnancies in general.

### The result, as a sentence with parameters in it

Generated from the fitted values above, so every number comes from a cell output. It names the
mid-August visit, because that is what makes shorter windows show a rise in messages per day
(step 5).

In [ ]:
d_b, d_a = daily["before"], daily["after"]
g_b, g_a = (gap_fits[s]["fits"]["lognorm"]["params"] for s in SIDES)
result_sentence = (
    f"Before the positive test we sent on average μ = {d_b['nb_mu']:.0f} messages a day "
    f"(negative binomial, r = {d_b['nb_r']:.1f}), afterwards μ = {d_a['nb_mu']:.0f} (r = {d_a['nb_r']:.1f}); "
    f"a difference a random split of the same days gives often (p = {null_daily.p_value():.2f}); "
    f"shorter windows show a rise only because we were together, and quiet, in mid-August. "
    f"Gaps within bursts stayed lognormal with median parameter e^μ = {g_b['scale']:.0f} s → {g_a['scale']:.0f} s "
    f"(σ = {g_b['s']:.2f} before, {g_a['s']:.2f} after). "
    f"The tempo of our conversations stayed the same; the content changed: fatigue and nausea "
    f"mentions went from {before_rate:.2f} to {after_rate:.2f} a day."
)
print(result_sentence)